# M03 — LCEL 與 Runnable 管線

本 notebook 對應 `README.md`，逐格執行即可。

目標：把 M01 的對話模型、M02 的 Prompt 模板，用一個 `|` 串成
一條可組合、可並行、自動支援串流與批次的 Runnable 管線。

## 1. 環境準備

載入共用 helper，取得供應商無關的 `model`（M01 已介紹）。
這格沒有可見輸出，成功跑完即可往下。

In [ ]:
# Load shared helpers so every notebook stays provider-agnostic.
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parents[1] / "_shared"))
from course_utils import get_model, load_env

load_env()
model = get_model()

## 2. 基本三段管線：prompt | model | StrOutputParser

這是 LCEL 的核心句型。`|` 把三個組件焊成一條鏈：

- `prompt` 吃字典、吐訊息（M02 學過）
- `model` 吃訊息、吐 `AIMessage`（M01 學過）
- `StrOutputParser` 吃 `AIMessage`、吐純文字 `str`（這個模組的新組件）

串好之後 `chain` 本身就是一個 Runnable，照樣用 `.invoke(...)`。

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一位{role}，回答精簡。"),
    ("human", "{question}"),
])

# The pipe operator wires output -> input, left to right.
chain = prompt | model | StrOutputParser()

result = chain.invoke({"role": "老師", "question": "一句話說明什麼是向量？"})
print(result)
# Expected output: a plain string (str), e.g.
# "向量是用一串數字來表示資料在多維空間中的位置與方向。"

### 對照：沒接 parser 會拿到什麼？

拿掉 `StrOutputParser`，鏈的輸出就是 `AIMessage` 而不是 `str`，
文字得自己從 `.content` 取。這就是 parser 幫我們省掉的一步。

In [ ]:
chain_no_parser = prompt | model
ai = chain_no_parser.invoke({"role": "老師", "question": "一句話說明什麼是向量？"})
print(type(ai).__name__)   # Expected output: AIMessage
print(ai.content)          # Expected output: 同上那段文字（需手動取 .content）

## 3. RunnableParallel：同一輸入，並行產生「標題」與「摘要」

我們有一段文章，想同時要「一個標題」和「一句摘要」。
這兩件事彼此獨立，適合並行。`RunnableParallel` 讓兩條子鏈吃**同一份輸入**、
同時跑，最後回傳一個字典 `{"title": ..., "summary": ...}`。

注意：在 `|` 旁邊直接寫一個 `dict`，LangChain 會自動把它當成 `RunnableParallel`，
所以下面兩種寫法等價，這裡用顯式的 `RunnableParallel` 讓概念更清楚。

In [ ]:
from langchain_core.runnables import RunnableParallel

title_prompt = ChatPromptTemplate.from_messages([
    ("human", "為以下文章想一個 10 字以內的標題：\n\n{article}"),
])
summary_prompt = ChatPromptTemplate.from_messages([
    ("human", "用一句話摘要以下文章：\n\n{article}"),
])

# Two independent sub-chains, both take the same {"article": ...} input.
title_chain = title_prompt | model | StrOutputParser()
summary_chain = summary_prompt | model | StrOutputParser()

parallel = RunnableParallel(title=title_chain, summary=summary_chain)

article = (
    "LangChain 的 LCEL 用一個管線運算子，把 prompt、模型與輸出解析器"
    "串成一條鏈。串好之後整條鏈自動支援單筆呼叫、串流與批次。"
)
out = parallel.invoke({"article": article})
print(out["title"])
print(out["summary"])
# Expected output: out is a dict with "title" and "summary" keys, both str.

## 4. RunnableLambda / @chain：在管線裡插入自訂步驟

鏈中間常需要做一點純 Python 的加工（清資料、改格式）。
把一般函式接進管線有兩種方式：

- 直接在 `|` 旁邊放函式，LangChain 會自動包成 `RunnableLambda`。
- 用 `RunnableLambda(fn)` 顯式包裝。

下面把上一格的並行結果，接一個自訂函式排版成一段文字。

In [ ]:
from langchain_core.runnables import RunnableLambda

def format_card(parts: dict) -> str:
    # Plain Python: take the parallel dict and lay it out as a card.
    return f"【{parts['title'].strip()}】\n{parts['summary'].strip()}"

# parallel -> custom function. The bare function is auto-wrapped as a Runnable;
# wrapping it explicitly with RunnableLambda makes the intent obvious.
card_chain = parallel | RunnableLambda(format_card)

print(card_chain.invoke({"article": article}))
# Expected output:
# 【<標題>】
# <摘要>

### `@chain`：把一個函式直接變成一條鏈

當自訂邏輯比較長、想當成獨立、可重用的鏈時，用 `@chain` 裝飾器最直接。
被裝飾的函式就變成一個 Runnable，照樣 `.invoke(...)`，也能再用 `|` 接到別的鏈。

In [ ]:
from langchain_core.runnables import chain

@chain
def shout(text: str) -> str:
    # A tiny custom chain: uppercase + emphasis.
    return text.upper() + " !!!"

# Compose the basic text chain with our custom @chain step.
loud_chain = chain_no_parser | StrOutputParser() | shout
print(loud_chain.invoke({"role": "助理", "question": "say hello in english"}))
# Expected output: the model's reply, upper-cased, ending with " !!!"

## 5. 同一條鏈，免費取得 .stream 與 .batch

重點來了：我們從頭到尾沒寫過任何串流或批次邏輯。
但因為「鏈本身就是 Runnable」，這兩個能力整條繼承。

### 5.1 串流：一塊一塊吐字

`.stream(...)` 回傳一個產生器，模型邊生成、我們邊印，適合即時顯示。

In [ ]:
# Stream the basic chain token-by-token (chunk-by-chunk).
for chunk in chain.stream({"role": "老師", "question": "用三句話介紹台灣。"}):
    print(chunk, end="", flush=True)   # chunk is str because of StrOutputParser
print()
# Expected output: the answer appears progressively, then a trailing newline.

### 5.2 批次：一次餵多筆輸入

`.batch([...])` 一次處理一整批輸入，回傳一個對齊順序的結果清單。
不用自己寫迴圈，LangChain 會並行處理這批請求。

In [ ]:
questions = [
    {"role": "老師", "question": "什麼是 token？"},
    {"role": "老師", "question": "什麼是 embedding？"},
    {"role": "老師", "question": "什麼是 prompt？"},
]
answers = chain.batch(questions)
for q, a in zip(questions, answers):
    print(f"Q: {q['question']}\nA: {a}\n")
# Expected output: a list of 3 str answers, in the same order as `questions`.

## 6. RunnablePassthrough：保留原輸入、在旁邊加工

有時你想把「原始輸入」原封不動往下傳，同時又生出一個衍生欄位。
`RunnablePassthrough` 就是那個「原樣傳遞」的環節。

下面用 `RunnableParallel`：一條分支用 passthrough 留住原句，
另一條分支請模型翻成英文。輸出同時含原句與譯文。

In [ ]:
from langchain_core.runnables import RunnablePassthrough

translate_prompt = ChatPromptTemplate.from_messages([
    ("human", "把這句翻成英文，只回譯文：{text}"),
])
translate_chain = translate_prompt | model | StrOutputParser()

# "original" keeps the raw input as-is; "english" is the derived translation.
enrich = RunnableParallel(
    original=RunnablePassthrough(),
    english=(lambda x: {"text": x}) | translate_chain,
)

print(enrich.invoke("今天天氣很好"))
# Expected output: {'original': '今天天氣很好', 'english': '<English translation>'}

## 🧪 練習 1：加一個字數統計分支

在第 3 節的 `parallel` 上，再加一條分支 `length`，
用 `RunnableLambda` 算出 `article` 的字數（提示：分支拿到的是整包輸入字典，
取 `x["article"]` 再 `len(...)`）。讓輸出變成
`{"title": ..., "summary": ..., "length": <int>}`。

In [ ]:
# TODO: 在這裡完成你的 parallel_with_length
# from langchain_core.runnables import RunnableParallel, RunnableLambda
# parallel_with_length = RunnableParallel(
#     title=title_chain,
#     summary=summary_chain,
#     length=RunnableLambda(lambda x: len(x["article"])),
# )
# print(parallel_with_length.invoke({"article": article}))

## 🧪 練習 2：把整條卡片鏈串流出來

第 4 節的 `card_chain` 也是一個 Runnable，所以一樣能 `.stream(...)`。
試著用 `for chunk in card_chain.stream({"article": article}): ...` 印出來，
觀察「最後一個自訂函式步驟」對串流行為的影響
（提示：函式步驟要等輸入到齊才動，串流的顆粒度會因此改變）。

In [ ]:
# TODO: 嘗試 card_chain.stream(...) 並印出結果，
# 比較它和第 5.1 節純文字鏈的串流體驗有何不同。

## 小結 & 下一步

這個模組你學會了：

- 用 `|` 把 `prompt | model | StrOutputParser()` 串成一條 Runnable 管線。
- 用 `RunnableParallel` 開並行分支、`RunnablePassthrough` 原樣傳遞、
  `RunnableLambda` / `@chain` 插自訂步驟。
- 因為鏈本身就是 Runnable，整條鏈自動繼承 `.invoke` / `.stream` / `.batch`。

三大好處一次到位：可組合、可並行、自動串流與批次。

下一個模組 **M04 — 工具與工具呼叫**：讓模型能呼叫你定義的函式去查資料、做計算，
這是邁向 Agent 的第一步。